# 📚 学术研究选题规划工具
## Academic Research Topic Planning Tool

**面向硕士生、博士生和高校教师的智能选题规划系统**

本工具基于 Claude claude-opus-4-6 大模型，通过四步分析流程，为学术研究人员提供专业的选题规划建议：

| 步骤 | 功能 | 说明 |
|------|------|------|
| 1️⃣ | **学术热点分析** | 识别领域内 5-8 个重要研究热点，评估趋势与重要性 |
| 2️⃣ | **研究缺口检测** | 发现现有研究的空白与不足，定位高价值填补点 |
| 3️⃣ | **多维选题树** | 从理论/方法/应用/交叉四个维度构建候选选题 |
| 4️⃣ | **研究价值评级** | 对每个选题进行多维评分，给出 ★ 评级和推荐建议 |

**输出：** 完整的选题建议报告（Markdown / JSON）

## 安装依赖 / Install Dependencies

In [ ]:
%pip install anthropic pydantic rich --quiet

## 配置 API Key / Configure API Key

In [ ]:
import os

# 方式1：直接设置（不推荐在生产环境使用）
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# 方式2：从环境变量读取（推荐）
if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("请输入 Anthropic API Key: ")

print("✅ API Key 已配置")

## 导入模块 / Import Module

In [ ]:
import sys
sys.path.insert(0, ".")

from academic_topic_planner import (
    AcademicTopicPlanner,
    TopicSuggestionReport,
    render_markdown,
    render_console,
)

print("✅ 模块加载成功")

## 示例1：计算机视觉领域选题规划

**场景：** 博士一年级学生，有 GPU 资源，对医学图像和自动驾驶感兴趣

In [ ]:
# 初始化规划器
planner = AcademicTopicPlanner(verbose=True)

# 配置研究领域和用户背景
FIELD = "计算机视觉"
CONTEXT = """
博士一年级学生，研究方向待定。
有扎实的深度学习基础，熟悉 PyTorch，拥有 4 块 A100 GPU。
导师要求3年内毕业，希望在顶会（CVPR/ICCV/ECCV）发表至少2篇论文。
对医学图像和自动驾驶两个应用方向感兴趣。
"""

# 生成完整报告（约需3-5分钟）
report = planner.generate_report(field=FIELD, context=CONTEXT.strip())

## 查看分析结果 / View Results

In [ ]:
# 控制台摘要
render_console(report)

In [ ]:
# 查看学术热点详情
print(f"\n🔥 共识别到 {len(report.hotspots)} 个学术热点：\n")
for h in report.hotspots:
    trend_arrow = {"rising": "📈", "stable": "➡️", "declining": "📉"}.get(h.trend.value, "")
    print(f"  {trend_arrow} [{h.importance_score}/10] {h.topic}")
    print(f"     关键词: {', '.join(h.core_keywords[:4])}")
    print()

In [ ]:
# 查看研究缺口
print(f"\n🔬 共检测到 {len(report.research_gaps)} 个研究缺口：\n")
for g in report.research_gaps:
    print(f"  • {g.gap_title}")
    print(f"    类型: {g.gap_type.value} | 难度: {g.difficulty.value} | 影响: {g.potential_impact.value}")
    print()

In [ ]:
# 查看研究价值评级（排序后）
def stars(n): return "★" * n + "☆" * (5 - n)

print("\n⭐ 研究价值评级（从高到低）：\n")
sorted_ratings = sorted(report.value_ratings, key=lambda r: r.stars, reverse=True)
for r in sorted_ratings:
    top_badge = " 🌟 TOP" if r.topic_id in report.top_recommendations else ""
    print(f"  {stars(r.stars)} {r.topic_title}{top_badge}")
    print(f"     推荐: {r.recommendation.value}")
    print(f"     创新:{stars(r.novelty)} 可行:{stars(r.feasibility)} "
          f"影响:{stars(r.impact)} 基金:{stars(r.funding_potential)}")
    print()

## 保存完整报告 / Save Full Report

In [ ]:
# 保存 Markdown 报告
md_content = render_markdown(report)
with open("topic_report_cv.md", "w", encoding="utf-8") as f:
    f.write(md_content)
print("✅ Markdown 报告已保存: topic_report_cv.md")

# 保存 JSON 数据
with open("topic_report_cv.json", "w", encoding="utf-8") as f:
    f.write(report.model_dump_json(indent=2, ensure_ascii=False))
print("✅ JSON 数据已保存: topic_report_cv.json")

In [ ]:
# 在 Notebook 中渲染 Markdown 报告预览
from IPython.display import Markdown, display
display(Markdown(md_content[:3000] + "\n\n*...（报告已截断，完整内容见 topic_report_cv.md）*"))

## 示例2：自定义领域快速规划

只需两行代码即可为任意领域生成选题报告：

In [ ]:
# 自定义你的领域和背景
my_field = "量子计算"  # 修改为你的研究领域
my_context = "理论物理背景的副教授，目标申请国家自然科学基金面上项目"  # 描述你的背景

# 生成报告
my_report = planner.generate_report(field=my_field, context=my_context)
render_console(my_report)

## 分步骤使用 / Step-by-Step Usage

如果你想对中间结果进行定制化处理，可以分步调用各分析模块：

In [ ]:
field = "大语言模型安全"
context = "网络安全方向的博士生，熟悉对抗攻击，有大模型微调经验"

# Step 1: 热点分析
hotspots = planner.analyze_hotspots(field, context)
print(f"热点数量: {len(hotspots)}")

# Step 2: 缺口检测（利用步骤1的结果）
gaps = planner.detect_research_gaps(field, context, hotspots)
print(f"缺口数量: {len(gaps)}")

# Step 3: 选题树构建
topic_tree = planner.build_topic_tree(field, context, hotspots, gaps)
print(f"选题节点: {len(topic_tree)}")

# Step 4: 价值评级
ratings = planner.rate_topics(field, context, topic_tree)
print(f"评级数量: {len(ratings)}")

# 查看最高评级的选题
best = max(ratings, key=lambda r: r.stars)
print(f"\n最高评级选题: {best.topic_title} ({'★' * best.stars})")
print(f"推荐意见: {best.recommendation.value}")
print(f"优势: {best.strengths[0]}")

## 命令行使用方式 / CLI Usage

```bash
# 基本用法
python academic_topic_planner.py --field "计算机视觉"

# 带背景信息
python academic_topic_planner.py \
    --field "量子计算" \
    --context "博士生，关注量子机器学习，有量子力学基础"

# 保存完整报告
python academic_topic_planner.py \
    --field "行为经济学" \
    --context "经济学硕士，有实验经济学基础" \
    --output report.md \
    --json report.json

# 使用演示脚本（预设场景）
python demo.py --scenario nlp --save
```